In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# -----------------------------
# 1. Dataset Generate karna
# -----------------------------
np.random.seed(42)

n = 1000

data = pd.DataFrame({
    "age": np.random.randint(18, 60, n),
    "bmi": np.random.uniform(18, 35, n),
    "exercise_hours": np.random.uniform(0, 10, n),
    "junk_food_score": np.random.randint(1, 10, n),
    "sleep_hours": np.random.uniform(4, 10, n)
})

# Target logic (Healthy = 1, Unhealthy = 0)
data["healthy"] = (
    (data["bmi"] < 25).astype(int) +
    (data["exercise_hours"] > 4).astype(int) +
    (data["junk_food_score"] < 5).astype(int) +
    (data["sleep_hours"] > 6).astype(int)
)

data["healthy"] = (data["healthy"] >= 3).astype(int)

# -----------------------------
# 2. Dataset Save karna
# -----------------------------
data.to_csv("healthy_data.csv", index=False)
print("Dataset saved as healthy_data.csv")

# -----------------------------
# 3. Train-Test Split
# -----------------------------
X = data.drop("healthy", axis=1)
y = data["healthy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 4. Model Training
# -----------------------------
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# -----------------------------
# 5. Prediction & Accuracy
# -----------------------------
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# -----------------------------
# 6. Model Save karna
# -----------------------------
joblib.dump(model, "healthy_model.pkl")
print("Model saved as healthy_model.pkl")

Dataset saved as healthy_data.csv
Model Accuracy: 100.00%
Model saved as healthy_model.pkl


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

np.random.seed(42)

n = 1200

# -----------------------------
# 1. Dataset Generate (Noisy + Weak Pattern)
# -----------------------------
data = pd.DataFrame({
    "age": np.random.randint(18, 60, n),
    "bmi": np.random.uniform(18, 40, n),  # wider range
    "exercise_hours": np.random.uniform(0, 8, n),
    "junk_food_score": np.random.randint(1, 10, n),
    "sleep_hours": np.random.uniform(3, 10, n)
})

# Weak + noisy target
data["healthy"] = (
    0.3 * (data["bmi"] < 27).astype(int) +
    0.3 * (data["exercise_hours"] > 3).astype(int) +
    0.2 * (data["sleep_hours"] > 6).astype(int) +
    0.2 * np.random.randint(0, 2, n)  # random noise
)

data["healthy"] = (data["healthy"] > 0.5).astype(int)

# -----------------------------
# 2. Drift Create karna (Train vs Test difference)
# -----------------------------
train_data = data.iloc[:900].copy()
test_data = data.iloc[900:].copy()

# Drift: test data ka behavior change
test_data["bmi"] += np.random.uniform(5, 10, len(test_data))  # shift
test_data["exercise_hours"] -= np.random.uniform(1, 3, len(test_data))
test_data["junk_food_score"] += np.random.randint(1, 3, len(test_data))

# -----------------------------
# 3. Save Dataset
# -----------------------------
data.to_csv("bad_data_drift.csv", index=False)
print("Dataset saved as bad_data_drift.csv")

# -----------------------------
# 4. Split
# -----------------------------
X_train = train_data.drop("healthy", axis=1)
y_train = train_data["healthy"]

X_test = test_data.drop("healthy", axis=1)
y_test = test_data["healthy"]

# -----------------------------
# 5. Model Training
# -----------------------------
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# -----------------------------
# 6. Prediction & Accuracy
# -----------------------------
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy (Drifted Data): {accuracy * 100:.2f}%")

# -----------------------------
# 7. Save Model
# -----------------------------
joblib.dump(model, "bad_model_drift.pkl")
print("Model saved as bad_model_drift.pkl")

Dataset saved as bad_data_drift.csv
Model Accuracy (Drifted Data): 61.67%
Model saved as bad_model_drift.pkl
